In [100]:
# set the project root so notebook imports work from the notebooks folder
import sys
from pathlib import Path

PROJECT_ROOT = Path.cwd().resolve().parent.parent
if str(PROJECT_ROOT) not in sys.path:
    sys.path.append(str(PROJECT_ROOT))

DATA_ROOT = PROJECT_ROOT / "Data"

print(PROJECT_ROOT)
print(DATA_ROOT)
print(DATA_ROOT.exists())

/Users/quentin/Documents/Computer/School/CMU/Spring_2026/38616/Gated-Risk-MCP
/Users/quentin/Documents/Computer/School/CMU/Spring_2026/38616/Gated-Risk-MCP/Data
True


In [101]:
# import the SROIE loader and feature/label builders from the repo code
import importlib
import pandas as pd

from utils.data_utils import load_sroie_split, preview_record
import src.sroie.sroie_features as sroie_features

importlib.reload(sroie_features)

<module 'src.sroie.sroie_features' from '/Users/quentin/Documents/Computer/School/CMU/Spring_2026/38616/Gated-Risk-MCP/src/sroie/sroie_features.py'>

In [102]:
# load the SROIE train split and confirm the raw pipeline is alive
records = load_sroie_split("train", data_root=DATA_ROOT)

print(len(records))
preview_record(records[0])

712
doc_id: X00016469612
dataset: SROIE
split: train
image_path: /Users/quentin/Documents/Computer/School/CMU/Spring_2026/38616/Gated-Risk-MCP/Data/SROIE/0325updated.task1train(626p)/X00016469612.jpg
image object present: False
image_size: None
num_tokens: 44
num_boxes: 44
num_fields: 4
field keys: ['company', 'date', 'address', 'total']
metadata keys: []


In [103]:
# build the receipt-level SROIE feature table from the loaded records
sroie_df = sroie_features.sroie_feature_dataframe(records)

display(sroie_df.head())
print(sroie_df.shape)
print(sorted(sroie_df.columns))

,doc_id,dataset,split,img_width,img_height,n_tokens,n_boxes,ocr_char_count,ocr_word_count,company_present,...,amount_token_ratio,date_token_ratio,avg_token_len,avg_words_per_token,anchors_present_count,fields_present_count,all_fields_present,any_field_missing,ocr_is_empty,aspect_ratio
0,X00016469612,SROIE,train,463.0,1013.0,44,44,441,84,True,...,0.159091,0.000000,10.022727,1.909091,3,4,True,False,False,0.457058
1,X00016469619,SROIE,train,439.0,1004.0,48,48,637,102,True,...,0.229167,0.000000,13.270833,2.125000,2,4,True,False,False,0.437251
2,X00016469620,SROIE,train,459.0,949.0,54,54,668,121,True,...,0.222222,0.000000,12.370370,2.240741,2,4,True,False,False,0.483667
3,X00016469622,SROIE,train,461.0,933.0,60,60,525,105,True,...,0.266667,0.016667,8.750000,1.750000,3,4,True,False,False,0.494105
4,X00016469623,SROIE,train,463.0,1026.0,61,61,735,142,True,...,0.245902,0.000000,12.049180,2.327869,2,4,True,False,False,0.451267


(712, 38)
['address_in_ocr', 'address_len', 'address_present', 'all_fields_present', 'amount_token_ratio', 'anchors_present_count', 'any_field_missing', 'aspect_ratio', 'avg_token_len', 'avg_words_per_token', 'company_in_ocr', 'company_len', 'company_present', 'dataset', 'date_in_ocr', 'date_len', 'date_present', 'date_token_ratio', 'doc_id', 'exact_total_matches', 'fields_present_count', 'has_cash_anchor', 'has_date_anchor', 'has_total_anchor', 'img_height', 'img_width', 'n_amount_like_tokens', 'n_boxes', 'n_date_like_tokens', 'n_tokens', 'ocr_char_count', 'ocr_is_empty', 'ocr_word_count', 'split', 'token_box_ratio', 'total_in_ocr', 'total_len', 'total_present']


In [104]:
# sanity-check the main numeric feature groups for missingness and scale
display(
    sroie_df[
        [
            "img_width",
            "img_height",
            "aspect_ratio",
            "n_tokens",
            "n_boxes",
            "ocr_char_count",
            "ocr_word_count",
            "company_len",
            "date_len",
            "address_len",
            "total_len",
            "exact_total_matches",
            "n_amount_like_tokens",
            "n_date_like_tokens",
            "token_box_ratio",
            "amount_token_ratio",
            "date_token_ratio",
            "avg_token_len",
            "avg_words_per_token",
            "anchors_present_count",
            "fields_present_count",
        ]
    ].describe()
)

,img_width,img_height,aspect_ratio,n_tokens,n_boxes,ocr_char_count,ocr_word_count,company_len,date_len,address_len,...,exact_total_matches,n_amount_like_tokens,n_date_like_tokens,token_box_ratio,amount_token_ratio,date_token_ratio,avg_token_len,avg_words_per_token,anchors_present_count,fields_present_count
count,712.000000,712.000000,712.000000,712.000000,712.000000,712.000000,712.000000,712.000000,712.000000,712.000000,...,712.000000,712.000000,712.000000,712.000000,712.000000,712.000000,712.000000,712.000000,712.000000,712.000000
mean,1269.889045,2283.568820,0.511129,52.744382,52.744382,605.705056,111.411517,23.209270,9.004213,66.592697,...,2.127809,13.224719,0.196629,0.988764,0.229278,0.004237,11.768749,2.142303,2.497191,3.755618
std,1345.629413,1753.298861,0.118331,18.362251,18.362251,172.144405,33.476621,9.203269,2.469216,24.076177,...,1.486635,9.525895,0.447641,0.105477,0.095327,0.009749,2.612188,0.418657,0.576124,0.954284
min,436.000000,605.000000,0.263498,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,...,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000
25%,623.000000,1402.000000,0.422162,41.000000,41.000000,502.000000,89.750000,18.000000,8.000000,54.000000,...,1.000000,7.000000,0.000000,1.000000,0.160000,0.000000,9.878686,1.877841,2.000000,4.000000
50%,793.000000,1675.000000,0.497784,49.000000,49.000000,580.000000,106.000000,22.000000,10.000000,70.000000,...,2.000000,11.000000,0.000000,1.000000,0.225000,0.000000,11.716063,2.152681,3.000000,4.000000
75%,934.000000,2086.750000,0.577567,62.000000,62.000000,707.000000,128.000000,30.000000,10.000000,84.000000,...,3.000000,17.000000,0.000000,1.000000,0.281360,0.000000,13.861035,2.385052,3.000000,4.000000
max,4961.000000,7016.000000,0.971354,153.000000,153.000000,1270.000000,237.000000,49.000000,12.000000,135.000000,...,7.000000,76.000000,4.000000,1.000000,0.539007,0.056604,19.727273,3.545455,3.000000,4.000000


In [105]:
# inspect boolean feature rates to confirm recoverability and anchor signals look sensible
display(
    sroie_df[
        [
            "company_present",
            "date_present",
            "address_present",
            "total_present",
            "company_in_ocr",
            "date_in_ocr",
            "address_in_ocr",
            "total_in_ocr",
            "has_total_anchor",
            "has_date_anchor",
            "has_cash_anchor",
            "all_fields_present",
            "any_field_missing",
            "ocr_is_empty",
        ]
    ]
    .mean()
    .to_frame("fraction_true")
)

,fraction_true
company_present,0.939607
date_present,0.939607
address_present,0.938202
total_present,0.938202
company_in_ocr,0.917135
date_in_ocr,0.932584
address_in_ocr,0.880618
total_in_ocr,0.935393
has_total_anchor,0.987360
has_date_anchor,0.542135


In [106]:
# build the proxy verification labels from the feature table
sroie_labels_df = sroie_features.sroie_proxy_label_dataframe(sroie_df)

display(sroie_labels_df.head())
print(sroie_labels_df.shape)

,doc_id,company_hard,date_hard,address_hard,total_hard,low_ocr_support,proxy_risk_score,strict_high_risk,review_worthy,proxy_verify,proxy_high_risk
0,X00016469612,True,False,False,False,False,1,False,True,False,False
1,X00016469619,False,False,False,False,False,0,False,False,False,False
2,X00016469620,True,False,False,False,False,1,False,True,False,False
3,X00016469622,False,False,False,False,False,0,False,False,False,False
4,X00016469623,False,False,False,False,False,0,False,False,False,False


(712, 11)


In [107]:
# inspect proxy-label balance before using these labels downstream
display(
    sroie_labels_df[
        [
            "company_hard",
            "date_hard",
            "address_hard",
            "total_hard",
            "low_ocr_support",
            "proxy_verify",
            "proxy_high_risk",
        ]
    ]
    .mean()
    .to_frame("fraction_true")
)

display(sroie_labels_df["proxy_risk_score"].value_counts().sort_index())

,fraction_true
company_hard,0.082865
date_hard,0.067416
address_hard,0.119382
total_hard,0.064607
low_ocr_support,0.122191
proxy_verify,0.066011
proxy_high_risk,0.066011


proxy_risk_score
0    547
1    106
2     14
4     34
5     11
Name: count, dtype: int64

In [108]:
# rebuild proxy labels after revising the rules and inspect balance again
sroie_labels_df = sroie_features.sroie_proxy_label_dataframe(sroie_df)

display(
    sroie_labels_df[
        [
            "company_hard",
            "date_hard",
            "address_hard",
            "total_hard",
            "low_ocr_support",
            "proxy_verify",
            "proxy_high_risk",
        ]
    ]
    .mean()
    .to_frame("fraction_true")
)

display(sroie_labels_df["proxy_risk_score"].value_counts().sort_index())

,fraction_true
company_hard,0.082865
date_hard,0.067416
address_hard,0.119382
total_hard,0.064607
low_ocr_support,0.122191
proxy_verify,0.066011
proxy_high_risk,0.066011


proxy_risk_score
0    547
1    106
2     14
4     34
5     11
Name: count, dtype: int64

In [109]:
# combine features and proxy labels into one final SROIE modeling table
sroie_model_df = sroie_df.merge(sroie_labels_df, on="doc_id", how="left")

display(sroie_model_df.head())
print(sroie_model_df.shape)

,doc_id,dataset,split,img_width,img_height,n_tokens,n_boxes,ocr_char_count,ocr_word_count,company_present,...,company_hard,date_hard,address_hard,total_hard,low_ocr_support,proxy_risk_score,strict_high_risk,review_worthy,proxy_verify,proxy_high_risk
0,X00016469612,SROIE,train,463.0,1013.0,44,44,441,84,True,...,True,False,False,False,False,1,False,True,False,False
1,X00016469619,SROIE,train,439.0,1004.0,48,48,637,102,True,...,False,False,False,False,False,0,False,False,False,False
2,X00016469620,SROIE,train,459.0,949.0,54,54,668,121,True,...,True,False,False,False,False,1,False,True,False,False
3,X00016469622,SROIE,train,461.0,933.0,60,60,525,105,True,...,False,False,False,False,False,0,False,False,False,False
4,X00016469623,SROIE,train,463.0,1026.0,61,61,735,142,True,...,False,False,False,False,False,0,False,False,False,False


(712, 48)


In [110]:
# inspect a few high-risk receipts to make sure the proxy labels are not obviously broken
high_risk_ids = (
    sroie_model_df
    .sort_values(["proxy_high_risk", "proxy_risk_score"], ascending=[False, False])["doc_id"]
    .head(5)
    .tolist()
)

for doc_id in high_risk_ids:
    record = next(r for r in records if r.doc_id == doc_id)
    print("=" * 80)
    print(doc_id)
    print(record.fields)
    print(record.ocr_tokens[:25])

display(
    sroie_model_df[
        [
            "doc_id",
            "proxy_risk_score",
            "proxy_verify",
            "proxy_high_risk",
            "company_in_ocr",
            "date_in_ocr",
            "address_in_ocr",
            "total_in_ocr",
            "exact_total_matches",
            "n_amount_like_tokens",
            "n_date_like_tokens",
            "ocr_is_empty",
        ]
    ]
    .sort_values(["proxy_high_risk", "proxy_risk_score"], ascending=[False, False])
    .head(10)
)

X51005433492(1)
{}
[]
X51005442378(1)
{}
['TQ FOR SHOPPING WITH MYNEWS.COM', 'PAGOH REST AND SERVICE AREA', 'LOT R1,KWS REHAT & RAWAT PAGOH ARAH UTARA,', 'SIMPANG AMPAT PAGOH,LEBUHRAYA UTARA SELATAN.,', '84600,PAGOH,JOHOR', 'MYNEWSCARELLNE : 180088 1231', 'MYNEWSCARELINE@MYNEWS.COM.MY', 'MYNEWS RETAIL SB(306295-X) FKA BISON STORES SB', 'TAX REG ID CBP 000709361664', 'QTY', 'PRICE', 'DISC', 'AMT', 'JALAN JALAN', '02', '1', '10.00', '0.00', '10.00 ZRL', 'SUB TOTAL', '10.00', 'GRAND TOTAL', '10.00', 'CASH 20.00 MYR', '20.00']
X51005442384(1)
{}
[]
X51005605333(1)
{}
[]
X51005676539(1)
{'company': 'SYARIKAT PERNIAGAAN GIN KEE', 'date': '10/01/2018', 'address': 'NO 290, JALAN AIR PANAS, SETAPAK, 53200, KUALA LUMPUR.', 'total': '23.32'}
[]


,doc_id,proxy_risk_score,proxy_verify,proxy_high_risk,company_in_ocr,date_in_ocr,address_in_ocr,total_in_ocr,exact_total_matches,n_amount_like_tokens,n_date_like_tokens,ocr_is_empty
34,X51005433492(1),5,True,True,False,False,False,False,0,0,0,True
58,X51005442378(1),5,True,True,False,False,False,False,0,9,0,False
64,X51005442384(1),5,True,True,False,False,False,False,0,0,0,True
119,X51005605333(1),5,True,True,False,False,False,False,0,0,0,True
144,X51005676539(1),5,True,True,False,False,False,False,0,0,0,True
169,X51005685355(2),5,True,True,False,False,False,False,0,0,0,True
172,X51005685357(2),5,True,True,False,False,False,False,0,0,0,True
293,X51006329395(2),5,True,True,False,False,False,False,0,8,0,False
522,X51007103681(1),5,True,True,False,False,False,False,0,3,1,False
566,X51007339118(1),5,True,True,False,False,False,False,0,0,0,True


In [111]:
# save the completed SROIE feature-plus-label table for later baseline and modeling notebooks
output_path = PROJECT_ROOT / "outputs" / "sroie_model_df.csv"

sroie_model_df.to_csv(output_path, index=False)
print(output_path)

/Users/quentin/Documents/Computer/School/CMU/Spring_2026/38616/Gated-Risk-MCP/outputs/sroie_model_df.csv


In [112]:
# reconstruct the current SROIE rule components directly from the model table
sroie_rule_df = sroie_model_df[[
    "doc_id",
    "company_hard",
    "date_hard",
    "address_hard",
    "total_hard",
    "low_ocr_support",
    "proxy_risk_score",
    "proxy_verify",
    "proxy_high_risk",
]].copy()

display(
    sroie_rule_df[
        [
            "company_hard",
            "date_hard",
            "address_hard",
            "total_hard",
            "low_ocr_support",
            "proxy_verify",
            "proxy_high_risk",
        ]
    ]
    .mean()
    .to_frame("fraction_true")
)

display(sroie_rule_df["proxy_risk_score"].value_counts().sort_index().to_frame("count"))

,fraction_true
company_hard,0.082865
date_hard,0.067416
address_hard,0.119382
total_hard,0.064607
low_ocr_support,0.122191
proxy_verify,0.066011
proxy_high_risk,0.066011


,count
proxy_risk_score,
0,547
1,106
2,14
4,34
5,11


In [113]:
# define a few alternative SROIE review targets so we can compare class balance
sroie_rule_df["review_v1_current"] = sroie_rule_df["proxy_verify"].astype(int)

sroie_rule_df["review_v2_any_hard_no_low_ocr"] = (
    sroie_rule_df[["company_hard", "date_hard", "address_hard", "total_hard"]]
    .sum(axis=1) >= 1
).astype(int)

sroie_rule_df["review_v3_two_or_more_signals_including_low_ocr"] = (
    sroie_rule_df[["company_hard", "date_hard", "address_hard", "total_hard", "low_ocr_support"]]
    .sum(axis=1) >= 2
).astype(int)

sroie_rule_df["review_v4_one_strong_or_two_total_date"] = (
    (
        sroie_rule_df["address_hard"]
        | sroie_rule_df["company_hard"]
        | (
            sroie_rule_df[["total_hard", "date_hard"]].sum(axis=1) >= 2
        )
        | (
            sroie_rule_df[["total_hard", "date_hard", "low_ocr_support"]].sum(axis=1) >= 2
        )
    )
).astype(int)

display(
    pd.DataFrame({
        "review_v1_current_rate": [sroie_rule_df["review_v1_current"].mean()],
        "review_v2_any_hard_no_low_ocr_rate": [sroie_rule_df["review_v2_any_hard_no_low_ocr"].mean()],
        "review_v3_two_or_more_signals_including_low_ocr_rate": [sroie_rule_df["review_v3_two_or_more_signals_including_low_ocr"].mean()],
        "review_v4_one_strong_or_two_total_date_rate": [sroie_rule_df["review_v4_one_strong_or_two_total_date"].mean()],
    })
)

,review_v1_current_rate,review_v2_any_hard_no_low_ocr_rate,review_v3_two_or_more_signals_including_low_ocr_rate,review_v4_one_strong_or_two_total_date_rate
0,0.066011,0.141854,0.082865,0.13764


In [114]:
# compare the alternative target distributions directly
for col in [
    "review_v1_current",
    "review_v2_any_hard_no_low_ocr",
    "review_v3_two_or_more_signals_including_low_ocr",
    "review_v4_one_strong_or_two_total_date",
]:
    print("=" * 80)
    print(col)
    display(sroie_rule_df[col].value_counts().sort_index().to_frame("count"))

review_v1_current


,count
review_v1_current,
0,665
1,47


review_v2_any_hard_no_low_ocr


,count
review_v2_any_hard_no_low_ocr,
0,611
1,101


review_v3_two_or_more_signals_including_low_ocr


,count
review_v3_two_or_more_signals_including_low_ocr,
0,653
1,59


review_v4_one_strong_or_two_total_date


,count
review_v4_one_strong_or_two_total_date,
0,614
1,98


In [115]:
# inspect how different the alternative labels really are
for col in [
    "review_v2_any_hard_no_low_ocr",
    "review_v3_two_or_more_signals_including_low_ocr",
    "review_v4_one_strong_or_two_total_date",
]:
    print("=" * 80)
    print(f"review_v1_current vs {col}")
    display(pd.crosstab(sroie_rule_df["review_v1_current"], sroie_rule_df[col]))

review_v1_current vs review_v2_any_hard_no_low_ocr


review_v2_any_hard_no_low_ocr,0,1
review_v1_current,,
0,611,54
1,0,47


review_v1_current vs review_v3_two_or_more_signals_including_low_ocr


review_v3_two_or_more_signals_including_low_ocr,0,1
review_v1_current,,
0,653,12
1,0,47


review_v1_current vs review_v4_one_strong_or_two_total_date


review_v4_one_strong_or_two_total_date,0,1
review_v1_current,,
0,614,51
1,0,47


In [116]:
# inspect which rule patterns dominate the new v4-only receipts
v4_only_df = sroie_model_df.loc[
    (sroie_rule_df["review_v4_one_strong_or_two_total_date"] == 1)
    & (sroie_rule_df["review_v1_current"] == 0),
    [
        "doc_id",
        "company_hard",
        "date_hard",
        "address_hard",
        "total_hard",
        "low_ocr_support",
        "n_tokens",
        "ocr_word_count",
        "n_amount_like_tokens",
        "exact_total_matches",
        "company_in_ocr",
        "date_in_ocr",
        "address_in_ocr",
        "total_in_ocr",
    ],
].copy()

print(v4_only_df.shape)
display(v4_only_df.head(20))

(51, 14)


,doc_id,company_hard,date_hard,address_hard,total_hard,low_ocr_support,n_tokens,ocr_word_count,n_amount_like_tokens,exact_total_matches,company_in_ocr,date_in_ocr,address_in_ocr,total_in_ocr
0,X00016469612,True,False,False,False,False,44,84,7,3,False,True,True,True
2,X00016469620,True,False,False,False,False,54,121,12,0,False,True,True,True
13,X51005268400,False,False,True,False,False,60,95,13,5,True,True,False,True
24,X51005361898,False,False,True,False,False,66,126,17,2,True,True,False,True
27,X51005361907(1),True,False,False,False,False,94,180,33,2,False,True,True,True
28,X51005361907,True,False,False,False,False,94,180,33,2,False,True,True,True
36,X51005433494,False,False,True,False,False,39,98,9,3,True,True,False,True
49,X51005442327,False,False,True,False,False,43,90,5,1,True,True,False,True
50,X51005442333,False,False,True,False,False,43,83,5,1,True,True,False,True
51,X51005442338,False,False,True,False,False,63,104,18,2,True,True,False,True


In [117]:
# summarize which hard signals are most common among the extra v4 positives
display(
    v4_only_df[
        [
            "company_hard",
            "date_hard",
            "address_hard",
            "total_hard",
            "low_ocr_support",
        ]
    ]
    .mean()
    .to_frame("fraction_true")
)

,fraction_true
company_hard,0.235294
date_hard,0.019608
address_hard,0.745098
total_hard,0.000000
low_ocr_support,0.235294


In [118]:
# compare current positives vs v4-only positives structurally
current_positive_df = sroie_model_df.loc[
    sroie_rule_df["review_v1_current"] == 1,
    [
        "n_tokens",
        "ocr_word_count",
        "n_amount_like_tokens",
        "exact_total_matches",
        "company_hard",
        "date_hard",
        "address_hard",
        "total_hard",
        "low_ocr_support",
    ]
]

v4_only_compare_df = v4_only_df[
    [
        "n_tokens",
        "ocr_word_count",
        "n_amount_like_tokens",
        "exact_total_matches",
        "company_hard",
        "date_hard",
        "address_hard",
        "total_hard",
        "low_ocr_support",
    ]
]

print("current positives")
display(current_positive_df.mean(numeric_only=True).to_frame("mean"))

print("v4-only added positives")
display(v4_only_compare_df.mean(numeric_only=True).to_frame("mean"))

current positives


,mean
n_tokens,42.765957
ocr_word_count,90.148936
n_amount_like_tokens,10.553191
exact_total_matches,0.042553
company_hard,1.000000
date_hard,0.957447
address_hard,1.000000
total_hard,0.957447
low_ocr_support,0.234043


v4-only added positives


,mean
n_tokens,50.529412
ocr_word_count,103.313725
n_amount_like_tokens,12.156863
exact_total_matches,1.843137
company_hard,0.235294
date_hard,0.019608
address_hard,0.745098
total_hard,0.000000
low_ocr_support,0.235294


In [119]:
# finalize the SROIE target definitions we want to carry forward
sroie_model_df["strict_high_risk"] = sroie_rule_df["review_v1_current"].astype(int)
sroie_model_df["review_worthy"] = sroie_rule_df["review_v4_one_strong_or_two_total_date"].astype(int)

display(
    pd.DataFrame({
        "strict_high_risk_rate": [sroie_model_df["strict_high_risk"].mean()],
        "review_worthy_rate": [sroie_model_df["review_worthy"].mean()],
    })
)

,strict_high_risk_rate,review_worthy_rate
0,0.066011,0.13764


In [120]:
# inspect overlap between the strict and broader review targets
display(pd.crosstab(sroie_model_df["strict_high_risk"], sroie_model_df["review_worthy"]))

review_worthy,0,1
strict_high_risk,,
0,614,51
1,0,47


In [121]:
# save the updated SROIE modeling table with both targets included
output_path = PROJECT_ROOT / "outputs" / "sroie_model_df.csv"
sroie_model_df.to_csv(output_path, index=False)

print(output_path)
print(sroie_model_df.shape)
display(
    sroie_model_df[
        [
            "doc_id",
            "strict_high_risk",
            "review_worthy",
            "company_hard",
            "date_hard",
            "address_hard",
            "total_hard",
            "low_ocr_support",
        ]
    ].head(10)
)

/Users/quentin/Documents/Computer/School/CMU/Spring_2026/38616/Gated-Risk-MCP/outputs/sroie_model_df.csv
(712, 48)


,doc_id,strict_high_risk,review_worthy,company_hard,date_hard,address_hard,total_hard,low_ocr_support
0,X00016469612,0,1,True,False,False,False,False
1,X00016469619,0,0,False,False,False,False,False
2,X00016469620,0,1,True,False,False,False,False
3,X00016469622,0,0,False,False,False,False,False
4,X00016469623,0,0,False,False,False,False,False
5,X00016469669,0,0,False,False,False,False,True
6,X00016469672,0,0,False,False,False,False,False
7,X00016469676,0,0,False,False,False,False,True
8,X51005200938,0,0,False,False,False,False,False
9,X51005230617,0,0,False,False,False,False,False


In [122]:
from utils.data_utils import load_sroie_split
from src.sroie.sroie_features import sroie_feature_dataframe, sroie_proxy_label_dataframe

records = load_sroie_split("train", data_root=DATA_ROOT)
feature_df = sroie_feature_dataframe(records)
label_df = sroie_proxy_label_dataframe(feature_df)

print(label_df[["strict_high_risk", "review_worthy", "proxy_verify", "proxy_high_risk"]].mean())
print(label_df.shape)
display(label_df.head())

strict_high_risk    0.066011
review_worthy       0.136236
proxy_verify        0.066011
proxy_high_risk     0.066011
dtype: float64
(712, 11)


,doc_id,company_hard,date_hard,address_hard,total_hard,low_ocr_support,proxy_risk_score,strict_high_risk,review_worthy,proxy_verify,proxy_high_risk
0,X00016469612,True,False,False,False,False,1,False,True,False,False
1,X00016469619,False,False,False,False,False,0,False,False,False,False
2,X00016469620,True,False,False,False,False,1,False,True,False,False
3,X00016469622,False,False,False,False,False,0,False,False,False,False
4,X00016469623,False,False,False,False,False,0,False,False,False,False


In [123]:
saved_df = pd.read_csv(PROJECT_ROOT / "outputs" / "sroie_model_df.csv")

compare_df = (
    saved_df[["doc_id", "strict_high_risk", "review_worthy"]]
    .rename(columns={
        "strict_high_risk": "saved_strict_high_risk",
        "review_worthy": "saved_review_worthy",
    })
    .merge(
        label_df[["doc_id", "strict_high_risk", "review_worthy"]],
        on="doc_id",
        how="inner",
    )
)

strict_mismatch_df = compare_df.loc[
    compare_df["saved_strict_high_risk"] != compare_df["strict_high_risk"]
].copy()

print(strict_mismatch_df.shape)
display(strict_mismatch_df.head(20))

(0, 5)


,doc_id,saved_strict_high_risk,saved_review_worthy,strict_high_risk,review_worthy


In [124]:
from src.sroie.sroie_rule_gate import run_rule_inference

rule_out = run_rule_inference(split="train")
print(rule_out.shape)
print(rule_out["action"].value_counts())
display(rule_out)

(712, 4)
action
auto_accept       615
review             50
human_required     47
Name: count, dtype: int64


,doc_id,strict_high_risk,review_worthy,action
0,X00016469612,False,True,review
1,X00016469619,False,False,auto_accept
2,X00016469620,False,True,review
3,X00016469622,False,False,auto_accept
4,X00016469623,False,False,auto_accept
...,...,...,...,...
707,X51008164997,False,False,auto_accept
708,X51008164998,False,False,auto_accept
709,X51008164999,False,False,auto_accept
710,X51009453801,False,False,auto_accept


In [125]:
from utils.data_utils import load_sroie_split
from src.sroie.sroie_features import sroie_feature_dataframe, sroie_proxy_label_dataframe

records = load_sroie_split("train", data_root=DATA_ROOT)
feature_df = sroie_feature_dataframe(records)
label_df = sroie_proxy_label_dataframe(feature_df)

print(label_df[["strict_high_risk", "review_worthy"]].sum())
print(label_df[["strict_high_risk", "review_worthy"]].mean())

strict_high_risk    47
review_worthy       97
dtype: int64
strict_high_risk    0.066011
review_worthy       0.136236
dtype: float64
